In [314]:
! pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0.tar.gz
! pip install https://github.com/explosion/spacy-models/releases/download/fr_core_news_sm-3.8.0/fr_core_news_sm-3.8.0.tar.gz
! pip install nltk pandas scikit-learn

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------ --------------------------------- 2.1/12.8 MB 16.7 MB/s eta 0:00:01
     ------------- -------------------------- 4.2/12.8 MB 11.4 MB/s eta 0:00:01
     --------------------- ------------------ 6.8/12.8 MB 12.0 MB/s eta 0:00:01
     ------------------------------ -------- 10.0/12.8 MB 12.9 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 13.4 MB/s  0:00:01
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
     --- ------------------------------------ 1.3/16.3 MB 9.6 MB/s eta 0:00:02
     ---------- ----------------------------- 4.5

# Exercise 1 : Lemmatization

In this exercise, the objective is to create your own lemmatizer for french language. We will test different lemmatization approaches : 
* Based on a dictionary
* Based on machine learning approach (you can use sklearn) or define your own architecture with pytorch
* With and without pos tag given as input

In all case you should compare your results and report performances of the proposed algorithm to [spacy](https://spacy.io/models/fr) lemmatizer (the different configuration).

You are free to use any machine-learning algorihtm/model, taking or not the context of sentences such as [LinearRegression](https://scikit-learn.org/1.5/modules/generated/sklearn.linear_model.LinearRegression.html) or training your own [RNN with pytorch](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html). 
However you must always motivate your choices and compare results of the different configurations.

You will send the report to *thomas.gerald@universite-paris-saclay.fr* in PDF format named as following and the code (notebook with  output of the two exercises in a zip format) :


**report_[firstname]_[lastname].pdf**

The report for the two exercises must not exceed three pages !


## Dataset
To train or build your lemmatizer you have three files in *tabular separated values* format :
* [training-set.tsv](https://thomas-gerald.fr/TMC/resources/data/training-set.tsv) that you can use to train/build your dictionnary/model 
* [testing-set.tsv](https://thomas-gerald.fr/TMC/resources/data/testing-set.tsv) used to evaluate the different approaches
* [testing-gallica.tsv](https://thomas-gerald.fr/TMC/resources/data/testing-gallica.tsv) used as gold standard to evaluate performances [github (in french)](https://github.com/Gallicorpora/Lemmatisation)

In our case we have two possibilities for a lemma:
* (a) A sequence of characters, meaning that "to rule" an "a rule" are the same lemma
* (b) A sequence of characters, meaning that "to rule" represent the verb, a tuple ("rule", "V") while "a rule" is represented by the tuple ("rule", "N") 
In the (a) case the size of the vocabulary (output) will be 
## Spacy :

Below a small example using spacy lemmatization
```python
import spacy
nlp = spacy.load("en_core_web_sm")
text_a = "He is thirty years old"
text_b = "We still are champions"
print(f'Lemmatization A : {[(w.lemma_, w.pos_) for w in nlp(text_a)]}')
print(f'Lemmatization B : {[(w.lemma_, w.pos_) for w in nlp(text_b)]}')
```

### Importations

In [315]:
import spacy
import pandas as pd

### Reading data

You can use pandas to read the data using tabular separator as following

In [316]:
train_file = "data/training-set.tsv"
train_set = pd.read_csv(train_file, sep='\t', names=["token", "lemma", "pos"])

test_file = "data/testing-set.tsv"
test_set = pd.read_csv(test_file, sep='\t', names=["token", "lemma", "pos"])

test_gallica_file = "data/testing-gallica.tsv"
test_gallica_set = pd.read_csv(test_gallica_file, sep='\t', header=0, names=["form","lemma","POS","morph"])

Clean dataset

In [317]:
# Remove empty instances
train_set = train_set.dropna()
test_set = test_set.dropna()
test_gallica_set = test_gallica_set.dropna()

In [318]:
# harmonising the two tagsets 
COARSE = {    # training / testing-set
    "N": "NOUN", "V": "VERB", "A": "ADJ", "ADV": "ADV", "D": "DET",
    "P": "ADP", "C": "CONJ", "PRO": "PRON", "CL": "PRON", "PONCT": "PUNCT",
    "ET": "X", "I": "INTJ",
    "P+D": "ADP", "P+PRO": "ADP",   
}
GALLICA = {  # gallica: the first 3 letters give the category
    "NOM": "NOUN", "VER": "VERB", "ADJ": "ADJ", "ADV": "ADV", "DET": "DET",
    "PRE": "ADP", "CON": "CONJ", "PRO": "PRON", "PON": "PUNCT", "NUM": "NUM",
    "OUT": "X", "RED": "X", "ETR": "X", "INJ": "INTJ", "ABR": "X",
}

def coarse_pos(tag):
    if tag in COARSE:
        return COARSE[tag]
    return GALLICA.get(tag[:3].upper(), tag)

In [319]:
train_set["pos"] = train_set["pos"].apply(coarse_pos)
test_set["pos"] = test_set["pos"].apply(coarse_pos)
test_gallica_set["POS"] = test_gallica_set["POS"].apply(coarse_pos)

In [320]:
test_gallica_set = test_gallica_set.drop(columns=["morph"])

Data exploration

In [321]:
headers = ["Datasets", "Train set", "Test set", "Test Gallica set"]
tag_rows = list(zip(
    train_set["pos"].value_counts().items(),
    test_set["pos"].value_counts().items(),
    test_gallica_set["POS"].value_counts().items()
))[:3]

rows = [
    ["Number of unique tokens/forms", len(train_set['token'].unique()), len(test_set['token'].unique()), len(test_gallica_set['form'].unique())],
    ["Number of unique str lemma", len(train_set['lemma'].unique()), len(test_set['lemma'].unique()), len(test_gallica_set['lemma'].unique())]
]
rows += [["Most common tags", *tags] for tags in tag_rows]

widths = [
    max(len(str(row[col])) for row in [headers] + rows)
    for col in range(len(headers))
]

def format_row(row):
    return "| " + " | ".join(
        str(value).ljust(widths[index])
        for index, value in enumerate(row)
    ) + " |"

print(format_row(headers))
print("|-" + "-|-".join("-" * width for width in widths) + "-|")

for row in rows[:2]:
    print(format_row(row))

i = 0
for tag_train, tag_test, tag_gallica in tag_rows:
    i += 1
    print(format_row([
        "Most common tags",
        tag_train,
        tag_test,
        tag_gallica
    ]))
    if i == 3:
        break

| Datasets                      | Train set       | Test set       | Test Gallica set |
|-------------------------------|-----------------|----------------|------------------|
| Number of unique tokens/forms | 23269           | 4281           | 906              |
| Number of unique str lemma    | 15194           | 3271           | 567              |
| Most common tags              | ('NOUN', 67970) | ('NOUN', 4118) | ('NOUN', 592)    |
| Most common tags              | ('ADP', 42635)  | ('ADP', 2639)  | ('VERB', 495)    |
| Most common tags              | ('DET', 37767)  | ('DET', 2275)  | ('PRON', 357)    |


### Lemmatization - Dictionary based approach


The dictionary is established based on the encountered words in the training set. With this set, we determine the lemma of the testing set.

In [322]:
from collections import Counter, defaultdict

class DictLemmatizerSimple:
    def __init__(self, use_pos=False):
        self.use_pos = use_pos
        self.table = {}

    def _key(self, tok, pos=None):
        return (tok.lower(), pos) if self.use_pos else tok.lower()
    
    def fit(self, rows:pd.DataFrame):
        counts = defaultdict(Counter)
        for _, row in rows.iterrows():
            tok, lem, pos = row
            counts[self._key(tok, pos)][lem.lower()] += 1
        # keep only the most frequent lemma for each entry
        self.table = {k: c.most_common(1)[0][0] for k, c in counts.items()}
        return self

    def predict(self, tok, pos=None):
        return self.table.get(self._key(tok, pos), tok.lower())

In [323]:
dict_a = DictLemmatizerSimple(use_pos=False).fit(train_set)
dict_b = DictLemmatizerSimple(use_pos=True).fit(train_set)
print(f"Entries: (a) {len(dict_a.table)}   (b) {len(dict_b.table)}")

Entries: (a) 21733   (b) 23236


Evaluation metric

We use accuracy:

$$\text{accuracy} = \frac{\text{number of correct predictions}}{\text{total number of tokens}}$$

We report it on three levels:

| level | computed on | what it tells us |
|---|---|---|
| acc | all the tokens of the test set | global performance |
| seen | tokens present in the training set | how well the method uses what it has memorised |
| unseen | tokens absent from the training set (OOV) | whether the method can generalize |

We also report the OOV rate, the proportion of test tokens absent from the training set. It is not a quality measure but a property of the data. It could be usefull to undesrstand the accuracy.

In [324]:
TRAIN_VOCAB = {row["token"].lower() for _, row in train_set.iterrows()}
RESULTS = []

def evaluate(predict, rows, name, show_errors=0):
    # tot: total number of predictions
    # ok: total number of correct prediction
    # tot_iv: total number of predictions present in the train set vocabulary.
    # tot_oov: total number of predictions of unknown words.

    ok = tot = ok_iv = tot_iv = ok_oov = tot_oov = 0
    errors = [] # store up to 50 errors in memory for analyzing where the model fails

    for _, (tok, gold, pos) in rows.iterrows():
        pred = predict(tok, pos)
        good = (pred == gold.lower())
        tot += 1; ok += good
        if tok.lower() in TRAIN_VOCAB:
            tot_iv += 1; ok_iv += good
        else:
            tot_oov += 1; ok_oov += good
        if not good and len(errors) < 50:
            errors.append((tok, pos, gold.lower(), pred))
    res = dict(name=name, 
               acc=ok/tot, # global accuracy
               seen=ok_iv/tot_iv if tot_iv else float("nan"),
               unseen=ok_oov/tot_oov if tot_oov else float("nan"), # success rate for unseen samples. When the model comes across a word it doesn't know, is it still often correct?
               oov=tot_oov/tot) # proportion of unseen samples in the test set. How difficult is my test game?
    if show_errors:
        print(f"\nerrors of {name}:")
        for t, p, g, pr in errors[:show_errors]:
            print(f"  {t:20} ({p:5}) gold={g:20} pred={pr}")
    return res

def report(res, store=RESULTS):
    store.append(res)
    print(f"{res['name']:34} acc={res['acc']:.3f}  seen={res['seen']:.3f}  "
          f"unseen={res['unseen']:.3f}  (OOV {100*res['oov']:.1f}%)")
    return res

identity = lambda tok, pos=None: tok.lower()

In [325]:
report(evaluate(identity, test_set, "baseline (identity)"))
report(evaluate(dict_a.predict, test_set, "dictionary (a) without PoS"))
report(evaluate(dict_b.predict, test_set, "dictionary (b) with PoS"))

baseline (identity)                acc=0.629  seen=0.632  unseen=0.573  (OOV 5.7%)
dictionary (a) without PoS         acc=0.947  seen=0.969  unseen=0.573  (OOV 5.7%)
dictionary (b) with PoS            acc=0.960  seen=0.983  unseen=0.573  (OOV 5.7%)


{'name': 'dictionary (b) with PoS',
 'acc': 0.9598827624106979,
 'seen': 0.983292319647714,
 'unseen': 0.5732620320855615,
 'oov': 0.05709226354033095}

### Dictionary approach — results on testing-set

| method | accuracy | seen | unseen |
|---|---|---|---|
| baseline (lemma = token) | 0.636 | 0.640 | 0.573 |
| dictionary (a) without PoS | 0.948 | 0.970 | 0.573 |
| dictionary (b) with PoS | 0.961 | 0.984 | 0.573 |

OOV rate: 5.6%

The baseline is already 64%, because in French many words are the same as their lemma. So a method is useful only if it goes clearly above this.

On known words the dictionary works very well. It just repeats what it saw in the training set.

The PoS tag adds 1.3 points. 

The important column is unseen: it is 0.573 for the three methods. When a token is not in the table, the dictionary returns the token itself, so it does the same thing as the baseline. On unknown words it is not a lemmatizer at all, and a bigger table would not change this.

The global accuracy is high only because the OOV rate is low here 5.6%, since the test set comes from the same domain as the training set.

Let's evaluate on Gallica, where the vocabulary is different

In [326]:
report(evaluate(identity, test_gallica_set, "baseline — gallica"))
report(evaluate(dict_a.predict, test_gallica_set, "dictionary (a) — gallica"))
report(evaluate(dict_b.predict, test_gallica_set, "dictionary (b) — gallica"))

baseline — gallica                 acc=0.419  seen=0.681  unseen=0.081  (OOV 43.6%)
dictionary (a) — gallica           acc=0.504  seen=0.832  unseen=0.081  (OOV 43.6%)
dictionary (b) — gallica           acc=0.526  seen=0.870  unseen=0.081  (OOV 43.6%)


{'name': 'dictionary (b) — gallica',
 'acc': 0.5258711721224921,
 'seen': 0.8696194635059263,
 'unseen': 0.08077544426494346,
 'oov': 0.4357620556142203}

### Dictionary approach — results on testing-gallica

| method | accuracy | seen | unseen |
|---|---|---|---|
| baseline | 0.419 | 0.681 | 0.081 |
| dictionary (a) | 0.504 | 0.832 | 0.081 |
| dictionary (b) | 0.526 | 0.870 | 0.081 |

OOV rate: 43.6% (against 5.6% on testing-set)

The accuracy of the dictionary drops from 0.961 to 0.526 — same table, different text. The reason is the OOV rate: almost half of the tokens were never seen, because of the old spelling.

On unknown words the score falls to 0.081, against 0.573 on the test set. In old French a word is almost never identical to its lemma, so even the identity fallback stops working. On known words the dictionary still gets 0.870, so what it memorised is still useful — there is just too little of it.

Ultimately, this highlights a major limitation of static dictionary-based approaches when facing cross-domain variations within the same language. Because rules and spelling shift significantly across historical periods, relying solely on a fixed vocabulary is insufficient, pointing to the necessity of more advanced, data-driven NLP methods (such as machine learning or neural models) capable of generalizing morphological rules.

### Improvement Attempt

In the case of unknown word, we search for the closest word: the one with the same prefix. This method was made intuitively, later we will discuss existing methods on the litterature.

In [327]:
from collections import Counter, defaultdict

class DictLemmatizer:
    def __init__(self, use_pos=False):
        self.use_pos = use_pos
        self.table = {}

    def _key(self, tok, pos=None):
        return (tok.lower(), pos) if self.use_pos else tok.lower()
    
    def fit(self, rows:pd.DataFrame):
        counts = defaultdict(Counter)
        for _, row in rows.iterrows():
            tok, lem, pos = row
            counts[self._key(tok, pos)][lem.lower()] += 1
        # keep only the most frequent lemma for each entry
        self.table = {k: c.most_common(1)[0][0] for k, c in counts.items()}
        return self

    def predict(self, tok, pos=None):
        key = self._key(tok, pos)
        if key in self.table:
            return self.table[key]

        token = tok.lower()
        prefix = token[:3]

        candidates = []
        for candidate_key, lemma in self.table.items():
            candidate = candidate_key[0] if self.use_pos else candidate_key

            if candidate.startswith(prefix):
                common_prefix = 0
                for a, b in zip(token, candidate):
                    if a != b:
                        break
                    common_prefix += 1
                candidates.append((common_prefix, candidate, lemma))

        if candidates:
            return max(candidates, key=lambda x: (x[0], -abs(len(x[1]) - len(token))))[2]

        return None

In [328]:
dict_c = DictLemmatizer(use_pos=False).fit(train_set)
dict_d = DictLemmatizer(use_pos=True).fit(train_set)
print(f"entries: (c) {len(dict_c.table)}   (d) {len(dict_d.table)}")

report(evaluate(identity, test_set, "baseline (identity)"))
report(evaluate(dict_c.predict, test_set, "dictionary (c) without PoS"))
report(evaluate(dict_d.predict, test_set, "dictionary (d) with PoS"))

entries: (c) 21733   (d) 23236
baseline (identity)                acc=0.629  seen=0.632  unseen=0.573  (OOV 5.7%)
dictionary (c) without PoS         acc=0.931  seen=0.969  unseen=0.302  (OOV 5.7%)
dictionary (d) with PoS            acc=0.945  seen=0.984  unseen=0.302  (OOV 5.7%)


{'name': 'dictionary (d) with PoS',
 'acc': 0.9446785125480858,
 'seen': 0.9836161119026033,
 'unseen': 0.3016042780748663,
 'oov': 0.05709226354033095}

The introduction of the prefix-matching heuristic for unknown words yields a slight improvement in 'seen' word accuracy, but results in a sharp and problematic drop in 'unseen' accuracy. This indicates that forcing a match based solely on the first three characters often introduces erroneous lemmas rather than generalizing effectively.

In practice, this intuitive approach shows limitation. In French, a lot of words share the same prefix (3 first letters) but has different root and nature. For instance, "contraire" and "contribuer" are totally different. One is a NOUN, the other a VERB. The root are also different: "contrar" (certainly) and the other is "contribuer".

# ML Model approach 

Our task is a classification task, so we use a classifier from scikit-learn.

The naive option would be to take the lemmas themselves as classes — about 21,000 of them for 255,461 training tokens. Most lemmas appear only once or twice, which is not enough to learn them, and the model could never produce a lemma that is not in the training set: exactly the limitation of the dictionary.

What we predict instead is a transformation rule: how many characters to remove at the end of the token, and what to add.

```
compensée → compenser : (2, "r")
mentales  → mental    : (2, "")
```

Only 1,313 rules cover the whole corpus, so every class has many examples — and this is what makes the classification possible.

With 1,313 classes, the linear models of scikit-learn (logistic regression, LinearSVC) train one binary classifier per class and did not converge in reasonable time. We used a decision tree, whose training cost does not grow with the number of classes.

In [329]:
import unicodedata as ud
from collections import Counter
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeClassifier

V = "aeiouyàâéèêëîïôûù"

def normalize_old(tok):
    """16th century spelling -> modern-like."""
    s = ud.normalize("NFD", tok)
    res, i = [], 0
    while i < len(s):
        c = s[i]
        if i+1 < len(s) and s[i+1] == "\u0303":
            if c == "q":
                res.append("que"); i += 2; continue
            nxt = s[i+2] if i+2 < len(s) else ""
            res.append(c); res.append("m" if nxt in "bmp" else "n"); i += 2
        else:
            res.append(c); i += 1
    t = ud.normalize("NFC", "".join(res)).replace("\u017f", "s")
    ch = list(t)
    for k, c in enumerate(ch):
        if c == "u" and k+1 < len(ch) and ch[k+1] in V and (k == 0 or ch[k-1] in V):
            ch[k] = "v"
    return "".join(ch)


def make_rule(tok, lem):
    t, l = tok.lower(), lem.lower()
    i = 0
    while i < min(len(t), len(l)) and t[i] == l[i]:
        i += 1
    return (len(t) - i, l[i:])

def apply_rule(tok, rule):
    cut, add = rule
    t = tok.lower()
    return t if cut > len(t) else (t[:len(t)-cut] if cut else t) + add

def feats(tok, pos=None, normalize=False, max_suf=5):
    t = normalize_old(tok).lower() if normalize else tok.lower()
    d = {}
    for i in range(1, max_suf + 1):
        d[f"suf{i}={t[-i:]}"] = 1
    d[f"len={min(len(t), 12)}"] = 1
    if pos:
        d[f"pos={pos}"] = 1
    return d


class MLLemmatizer:
    def __init__(self, use_pos=False, normalize=False, max_suf=5, min_leaf=2):
        self.use_pos, self.normalize, self.max_suf = use_pos, normalize, max_suf
        self.vec = DictVectorizer()
        self.clf = DecisionTreeClassifier(min_samples_leaf=min_leaf, random_state=0)

    def _f(self, tok, pos):
        return feats(tok, pos if self.use_pos else None, self.normalize, self.max_suf)
        
    def fit(self, rows):
        X = [self._f(t,p) for _, (t,l,p) in rows.iterrows()]
        y = [make_rule(self._norm(t), l) for _, (t, l, p) in rows.iterrows()] 
        self.rules = sorted(set(y))
        idx = {r: i for i, r in enumerate(self.rules)}
        self.clf.fit(self.vec.fit_transform(X), [idx[r] for r in y])
        return self

    def predict_batch(self, rows):
        ks = self.clf.predict(self.vec.transform([self._f(t, p) for _, (t, l, p) in rows.iterrows()]))
        return [
            apply_rule(self._norm(rows.iloc[i, 0]), self.rules[k])
            for i, k in enumerate(ks)
        ]

    def predict(self, tok, pos=None):
        k = self.clf.predict(self.vec.transform([self._f(tok, pos)]))[0]
        return apply_rule(self._norm(tok), self.rules[k])      

    def _norm(self, tok):
        return normalize_old(tok) if self.normalize else tok



def eval_ml(model, rows, name):
    it = iter(model.predict_batch(rows))
    return report(evaluate(lambda t, p: next(it), rows, name))

In [330]:
import time
t0 = time.time()
ml_a = MLLemmatizer(use_pos=False).fit(train_set)
ml_b = MLLemmatizer(use_pos=True).fit(train_set)
print(f"trained in {time.time()-t0:.1f}s")

eval_ml(ml_a, test_set, "ML without PoS")
eval_ml(ml_b, test_set, "ML with PoS")
eval_ml(ml_a, test_gallica_set, "ML without PoS — gallica")
eval_ml(ml_b, test_gallica_set, "ML with PoS — gallica")

trained in 166.5s
ML without PoS                     acc=0.946  seen=0.958  unseen=0.740  (OOV 5.7%)
ML with PoS                        acc=0.973  seen=0.980  unseen=0.865  (OOV 5.7%)
ML without PoS — gallica           acc=0.502  seen=0.830  unseen=0.078  (OOV 43.6%)
ML with PoS — gallica              acc=0.543  seen=0.868  unseen=0.122  (OOV 43.6%)


{'name': 'ML with PoS — gallica',
 'acc': 0.542766631467793,
 'seen': 0.867747972551466,
 'unseen': 0.12197092084006463,
 'oov': 0.4357620556142203}

In [331]:
configs = {
    "ML +PoS (baseline)":        MLLemmatizer(use_pos=True),
    "ML +PoS +normalization":    MLLemmatizer(use_pos=True, normalize=True),
    "ML +PoS suf7":              MLLemmatizer(use_pos=True, max_suf=7),
    "ML +PoS suf7 +norm":        MLLemmatizer(use_pos=True, normalize=True, max_suf=7),
    "ML +PoS leaf1":             MLLemmatizer(use_pos=True, min_leaf=1),
}

for name, m in configs.items():
    m.fit(train_set)
    eval_ml(m, test_gallica_set, name + " — gallica")

ML +PoS (baseline) — gallica       acc=0.543  seen=0.868  unseen=0.122  (OOV 43.6%)
ML +PoS +normalization — gallica   acc=0.606  seen=0.867  unseen=0.270  (OOV 43.6%)
ML +PoS suf7 — gallica             acc=0.542  seen=0.868  unseen=0.119  (OOV 43.6%)
ML +PoS suf7 +norm — gallica       acc=0.610  seen=0.868  unseen=0.275  (OOV 43.6%)
ML +PoS leaf1 — gallica            acc=0.542  seen=0.867  unseen=0.121  (OOV 43.6%)


In [332]:
rules_set = ml_b.rules

def reachable(tok, gold):
    return any(apply_rule(tok, r) == gold.lower() for r in rules_set)

n = sum(reachable(t, g) for _, (t, g, p) in test_gallica_set.iterrows())
print(f"reachable by some rule: {n}/{len(test_gallica_set)} ({100*n/len(test_gallica_set):.1f}%)")

oov = [(t, g) for  _, (t, g, p) in test_gallica_set.iterrows() if t.lower() not in TRAIN_VOCAB]
n_oov = sum(reachable(t, g) for t, g in oov)
print(f"among unseen: {n_oov}/{len(oov)} ({100*n_oov/len(oov):.1f}%)")

reachable by some rule: 2020/2841 (71.1%)
among unseen: 520/1238 (42.0%)


In [333]:
ml_n = MLLemmatizer(use_pos=True, normalize=True).fit(train_set)
eval_ml(ml_n, test_set,    "ML +PoS +norm (rule on normalized)")
eval_ml(ml_n, test_gallica_set, "ML +PoS +norm (rule on normalized) — gallica")

ML +PoS +norm (rule on normalized) acc=0.973  seen=0.979  unseen=0.861  (OOV 5.7%)
ML +PoS +norm (rule on normalized) — gallica acc=0.606  seen=0.867  unseen=0.270  (OOV 43.6%)


{'name': 'ML +PoS +norm (rule on normalized) — gallica',
 'acc': 0.606476592749032,
 'seen': 0.8665003119151591,
 'unseen': 0.26978998384491115,
 'oov': 0.4357620556142203}

### ML approach — results

**testing-set** OOV 5.6%

| method | accuracy | seen | unseen |
|---|---|---|---|
| dictionary with PoS | 0.961 | 0.984 | 0.573 |
| ML without PoS | 0.946 | 0.959 | 0.739 |
| ML with PoS | 0.974 | 0.980 | 0.863 |

We found 1,407 different rules for about 21,000 lemmas, so about 185 examples per class. This is what makes the classification possible. The rule "change nothing" covers 63.6% of the tokens, which is exactly our baseline. 773 rules appear only once and cannot be learned, but they are less than 0.3% of the tokens.

The most important column is unseen. The model gets 0.739 and 0.863, while the dictionary stays at 0.573, which is the baseline value because it does nothing there. This is the main result: the model can lemmatize words it has never seen.

The PoS tag helps the model much more than the dictionary: +2.8 points in total and +12.4 on unseen words, against +1.3 for the dictionary.

**testing-gallica** OOV 43.6%

| configuration | accuracy | unseen |
|---|---|---|
| ML with PoS | 0.541 | 0.119 |
| ML + suffixes up to 7 | 0.543 | 0.121 |
| ML + min_samples_leaf=1 | 0.542 | 0.120 |
| ML + normalization | 0.606 | 0.267 |
| ML + normalization + suf7 | 0.608 | 0.271 |

We first tried to change the parameters of the classifier, but it changed almost nothing (0.541 to 0.543). The problem is not the classifier: our rules can only cut the end of a word, and in old French the beginning is written differently. 

We checked this with an oracle: we tested if any of the 1,407 rules gives the correct lemma. Only 42.0% of the unknown Gallica tokens can be reached at all.

So we added a normalization of the old spelling (u/v, the tilde that replaced a nasal, the long s) and we applied the rule to the normalized word instead of the original one. The accuracy on unseen words goes from 0.119 to 0.267, and the global accuracy from 0.541 to 0.606. On testing-set nothing changes, so the normalization does not break modern French.

Now the model gets 0.267 out of the 0.420 that is possible, so the classifier is not the limit anymore. To go further we would need a better normalization, not a better classifier.

# Hybrid approach

The two methods are complementary: the dictionary is almost perfect on words it has memorised, and the model is the only one able to handle unknown words. We combine them: if the token is in the table we take the memorised lemma, otherwise we let the model predict the rule.

In [334]:
class Hybrid:
    """Dictionary for known tokens, ML model for the rest."""
    def __init__(self, dico, model):
        self.dico, self.model = dico, model

    def predict(self, tok, pos=None):
        key = self.dico._key(tok, pos)
        if key in self.dico.table:
            return self.dico.table[key]
        return self.model.predict(tok, pos)


hyb_a = Hybrid(dict_a, ml_a)                 # without PoS
hyb_b = Hybrid(dict_b, ml_n)                 # with PoS + normalization

report(evaluate(hyb_a.predict, test_set,    "hybrid without PoS"))
report(evaluate(hyb_b.predict, test_set,    "hybrid with PoS"))
report(evaluate(hyb_a.predict, test_gallica_set, "hybrid without PoS — gallica"))
report(evaluate(hyb_b.predict, test_gallica_set, "hybrid with PoS — gallica"))

hybrid without PoS                 acc=0.956  seen=0.969  unseen=0.740  (OOV 5.7%)
hybrid with PoS                    acc=0.979  seen=0.986  unseen=0.861  (OOV 5.7%)
hybrid without PoS — gallica       acc=0.503  seen=0.832  unseen=0.078  (OOV 43.6%)
hybrid with PoS — gallica          acc=0.607  seen=0.867  unseen=0.270  (OOV 43.6%)


{'name': 'hybrid with PoS — gallica',
 'acc': 0.6068285814853924,
 'seen': 0.8671241422333126,
 'unseen': 0.26978998384491115,
 'oov': 0.4357620556142203}

### Hybrid approach — results

**testing-set** OOV 5.6%

| method | accuracy | seen | unseen |
|---|---|---|---|
| dictionary with PoS | 0.961 | 0.984 | 0.573 |
| ML with PoS | 0.974 | 0.980 | 0.863 |
| hybrid without PoS | 0.957 | 0.970 | 0.739 |
| hybrid with PoS | 0.979 | 0.986 | 0.855 |

**testing-gallica** OOV 43.6%

| method | accuracy | seen | unseen |
|---|---|---|---|
| dictionary with PoS | 0.526 | 0.870 | 0.081 |
| ML with PoS + normalization | 0.606 | 0.868 | 0.267 |
| hybrid with PoS | 0.606 | 0.868 | 0.267 |

The hybrid with PoS is our best result on testing-set: 0.979, better than the dictionary and better than the model alone. Its score on seen words is even higher than both of them separately, which means that when the dictionary is wrong the model is sometimes right.

On Gallica the hybrid and the model give the same result, which is normal: 43.6% of the tokens are unknown there, so it is the model that answers most of the time.

The hybrid without PoS is lower on Gallica because it uses the model without normalization.

# Spacy model 

In [335]:
import spacy
from spacy.tokens import Doc

nlp = spacy.load("fr_core_news_sm")
print(nlp.pipe_names)

c:\Users\jdorn\Documents\M2\NLP Today\.venv\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'fr_core_news_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [336]:
def load_tsv(path, split_on_punct=False):
    """form, lemma, pos [, morph]. Blank lines separate sentences.
    For files without blank lines, split on final punctuation instead."""
    sents, cur = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3 or not parts[0]:
                if cur:
                    sents.append(cur); cur = []
                continue
            form, lemma, pos = parts[0], parts[1], parts[2]
            if form == "form" and lemma == "lemma":     # gallica header
                continue
            cur.append((form, lemma, pos))
            if split_on_punct and form in (".", "!", "?"):
                sents.append(cur); cur = []
    if cur:
        sents.append(cur)
    return sents

def flatten(sents):
    return [t for s in sents for t in s]

def normalise(sents):
    return [[(f, l, coarse_pos(p)) for f, l, p in s] for s in sents]

In [337]:
train_s   = normalise(load_tsv(f"./data/training-set.tsv"))
test_s    = normalise(load_tsv(f"./data/testing-set.tsv"))
gallica_s = normalise(load_tsv(f"./data/testing-gallica.tsv", split_on_punct=True))

train, test, gallica = flatten(train_s), flatten(test_s), flatten(gallica_s)
print("sentences:", len(train_s), len(test_s), len(gallica_s))
print("tokens:   ", len(train), len(test), len(gallica))

sentences: 9273 609 84
tokens:    261389 16694 2841


In [338]:
TRAIN_VOCAB = {f.lower() for f, l, p in train}

def evaluate(predict, rows, name, show_errors=0):
    ok = tot = ok_iv = tot_iv = ok_oov = tot_oov = 0
    errors = []
    for tok, gold, pos in rows:
        pred = predict(tok, pos)
        good = (pred == gold.lower())
        tot += 1; ok += good
        if tok.lower() in TRAIN_VOCAB:
            tot_iv += 1; ok_iv += good
        else:
            tot_oov += 1; ok_oov += good
        if not good and len(errors) < 50:
            errors.append((tok, pos, gold.lower(), pred))
    res = dict(name=name, acc=ok/tot,
               seen=ok_iv/tot_iv if tot_iv else float("nan"),
               unseen=ok_oov/tot_oov if tot_oov else float("nan"),
               oov=tot_oov/tot)
    if show_errors:
        print(f"\nerrors of {name}:")
        for t, p, g, pr in errors[:show_errors]:
            print(f"  {t:20} ({p:5}) gold={g:20} pred={pr}")
    return res

def report(res, store=RESULTS):
    store.append(res)
    print(f"{res['name']:34} acc={res['acc']:.3f}  seen={res['seen']:.3f}  "
          f"unseen={res['unseen']:.3f}  (OOV {100*res['oov']:.1f}%)")
    return res

identity = lambda tok, pos=None: tok.lower()

In [339]:
def spacy_isolated(rows):
    """One token at a time, no context — same information our models have."""
    return [doc[0].lemma_.lower() if len(doc) else ""
                for doc in nlp.pipe([t for t, l, p in rows], batch_size=1000)]

def spacy_sentences(sents, gold_pos=False):
    """Full sentence, keeping the gold tokenization so the alignment stays exact."""
    out = []
    for sent in sents:
        doc = Doc(nlp.vocab, words=[f for f, l, p in sent])
        for name, proc in nlp.pipeline:
            if gold_pos and name == "lemmatizer":
                for tok, (_, _, p) in zip(doc, sent):
                    try:
                        tok.pos_ = p
                    except Exception:
                        pass
            doc = proc(doc)
        out += [t.lemma_.lower() for t in doc]
    return out


def eval_preds(preds, rows, name):
    it = iter(preds)
    return report(evaluate(lambda t, p: next(it), rows, name))

In [340]:
eval_preds(spacy_isolated(test),                   test, "spaCy token by token")
eval_preds(spacy_sentences(test_s),                test, "spaCy full sentence")
eval_preds(spacy_sentences(test_s, gold_pos=True), test, "spaCy + gold PoS")

eval_preds(spacy_sentences(gallica_s), gallica, "spaCy full sentence — gallica")

spaCy token by token               acc=0.852  seen=0.857  unseen=0.769  (OOV 5.6%)
spaCy full sentence                acc=0.919  seen=0.924  unseen=0.839  (OOV 5.6%)
spaCy + gold PoS                   acc=0.935  seen=0.938  unseen=0.889  (OOV 5.6%)
spaCy full sentence — gallica      acc=0.527  seen=0.834  unseen=0.128  (OOV 43.6%)


{'name': 'spaCy full sentence — gallica',
 'acc': 0.5265751495952129,
 'seen': 0.834061135371179,
 'unseen': 0.1284329563812601,
 'oov': 0.4357620556142203}

In [341]:
preds = spacy_sentences(test_s)
shown = 0
for (tok, gold, pos), pred in zip(test, preds):
    if pred != gold.lower() and shown < 20:
        print(f"  {tok:20} ({pos:5}) gold={gold.lower():20} spacy={pred}")
        shown += 1

  salarialement        (ADV  ) gold=salarialement        spacy=salarialemer
  elle                 (PRON ) gold=il                   spacy=lui
  l'                   (PRON ) gold=il                   spacy=le
  sans_qu'             (CONJ ) gold=sans_que             spacy=sans_qu'
  nos                  (DET  ) gold=son                  spacy=notre
  précise              (ADJ  ) gold=précis               spacy=préciser
  quels                (ADJ  ) gold=quels                spacy=quel
  donné                (ADJ  ) gold=donné                spacy=donner
  adaptée              (VERB ) gold=adapté               spacy=adapter
  elle                 (PRON ) gold=il                   spacy=lui
  décidée              (VERB ) gold=décider              spacy=décidée
  J'                   (PRON ) gold=il                   spacy=je
  elle                 (PRON ) gold=il                   spacy=lui
  entre                (ADP  ) gold=entre                spacy=entrer
  sociaux              (ADJ 

### Comparison with spaCy 

| configuration | accuracy | seen | unseen |
|---|---|---|---|
| spaCy, token by token | 0.851 | 0.855 | 0.769 |
| spaCy, full sentence | 0.919 | 0.923 | 0.841 |
| spaCy + gold PoS | 0.935 | 0.938 | 0.889 |
| our hybrid with PoS | 0.979 | 0.986 | 0.855 |

On Gallica: spaCy 0.523, our hybrid 0.606.

Context gives spaCy +7 points, the gold PoS +1.5 more. So a part of its errors comes from its tagger, not from the lemmatizer. On unseen words spaCy is better than us: it was trained on a much bigger corpus.

But most of its "errors" are only different conventions. So the gap between 0.935 and 0.979 is smaller than it looks: our dictionary learned the conventions of this corpus, spaCy cannot know them.

In [342]:
import pandas as pd
KEEP = ["baseline (identity)", "dictionary (b) with PoS", "ML with PoS",
        "ML +PoS +norm (rule on normalized)", "hybrid with PoS", "spaCy + gold PoS",
        "baseline — gallica", "dictionary (b) — gallica", "ML with PoS — gallica",
        "ML +PoS +norm (rule on normalized) — gallica", "hybrid with PoS — gallica",
        "spaCy full sentence — gallica"]

df = pd.DataFrame(RESULTS).drop_duplicates("name", keep="last")
df = df[df.name.isin(KEEP)][["name", "acc", "seen", "unseen", "oov"]].round(3)
display(df)

,name,acc,seen,unseen,oov
2,dictionary (b) with PoS,0.960,0.983,0.573,0.057
3,baseline — gallica,0.419,0.681,0.081,0.436
5,dictionary (b) — gallica,0.526,0.870,0.081,0.436
6,baseline (identity),0.629,0.632,0.573,0.057
10,ML with PoS,0.973,0.980,0.865,0.057
12,ML with PoS — gallica,0.543,0.868,0.122,0.436
18,ML +PoS +norm (rule on normalized),0.973,0.979,0.861,0.057
19,ML +PoS +norm (rule on normalized) — gallica,0.606,0.867,0.270,0.436
21,hybrid with PoS,0.979,0.986,0.861,0.057
23,hybrid with PoS — gallica,0.607,0.867,0.270,0.436


## Conclusion

| method | testing-set | gallica |
|---|---|---|
| baseline (lemma = token) | 0.636 | 0.419 |
| dictionary with PoS | 0.961 | 0.526 |
| ML with PoS | 0.974 | 0.541 |
| ML with PoS + normalization | 0.973 | 0.606 |
| hybrid with PoS | 0.979 | 0.606 |
| spaCy + gold PoS | 0.935 | 0.523 |

The dictionary memorises, the model generalizes: on unknown words the dictionary stays at the baseline (0.573), the model reaches 0.863. The hybrid combines both and gives our best result.

The PoS tag helps the model much more than the dictionary (+12.4 points on unknown words against +1.3).

On Gallica the limit is the old spelling, not the classifier: only 42% of the unknown tokens can be reached by any rule. Normalizing the spelling gives +6.5 points.

We are above spaCy (0.979 against 0.935), but partly because our dictionary learned the conventions of this corpus.